# Collecting Variables

In [1]:
# Parses instancias.txt: fleet, loader, mine-front, control, discharge and cycle-time parameters.
# Each parameter is found with a regular expression; the results go into the dictionary 'variaveis'.
import re

# Name of the instance file, read from the current working directory (run the notebook from dados/).
ARQ_INSTANCIAS = "instancias.txt"

# Dictionary to store the extracted variables
# Dictionary of extracted parameters: key = parameter name, value = list of parsed values.
variaveis = {}

# Reads the file until the end marker "/*" is found
# Lines of the header part of instancias.txt, without surrounding whitespace.
conteudo = []
# Read instancias.txt line by line and stop at the first line containing "/*" (the comment block after the parameters is not parsed).
with open(ARQ_INSTANCIAS, "r", encoding="utf-8") as f:
    for linha in f:
        if "/*" in linha:
            break
        conteudo.append(linha.strip())

# Joins the lines into one continuous text (to make the regex easier)
# One string with all header lines, so the regular expressions can span several lines (re.S).
texto = "\n".join(conteudo)

# === Regular expressions to capture the blocks ===

# Homogeneous truck fleet
# Homogeneous fleet block. Groups: (1) truck counts after "number of trucks", (2) waste/ore capacity pairs, (3) cycle-time labels.
# re.S lets "." cross line breaks and re.I ignores case.
match_homog = re.search(
    r"Cam\.?\s*Homogeneous\s*Fleet.*?number\s*of\s*trucks.*?:\s*([0-9;\s]+)"
    r".*?capacity.*?\(?.*?est[ée]ril\s*/\s*min[ée]rio\)?.*?:\s*([0-9/\s;]+)"
    r".*?cycle\s*time.*?:\s*([\w\s;]+)",
    texto, re.S | re.I
)
# Fill the homogeneous-fleet variables only when the block was found.
if match_homog:
    # Number of trucks per configuration: every integer found in group 1.
    variaveis["numero_caminhoes_homogeneo"] = [
        int(x) for x in re.findall(r"\d+", match_homog.group(1))
    ]
    # Split group 2 into "waste/ore" capacity pairs, e.g. "61/64; 120/130".
    caps = [c.strip() for c in match_homog.group(2).split(";") if c.strip()]
    esteril, minerio = [], []
    # Split each pair into waste (esteril) and ore (minerio) capacity values.
    for c in caps:
        e, m = c.split("/")
        esteril.append(int(e))
        minerio.append(int(m))
    # Store waste and ore capacities as two separate lists.
    variaveis["capaciti_caminhao_homogeneo_esteril"] = esteril
    variaveis["capaciti_caminhao_homogeneo_minerio"] = minerio
    # Cycle-time labels (low, avg, high) from group 3, split on ";".
    variaveis["tempo_ciclo_homogeneo"] = [
        x.strip() for x in match_homog.group(3).split(";") if x.strip()
    ]

# Heterogeneous truck fleet
# Heterogeneous fleet block: like the homogeneous one, plus a "Distribuicao" group (share of each truck type) before the cycle-time labels.
match_heterog = re.search(
    r"Cam\.?\s*Heterogeneous\s*Fleet.*?number\s*of\s*trucks.*?:\s*([0-9;\s]+)"
    r".*?capacity.*?\(?.*?est[ée]ril\s*/\s*min[ée]rio\)?.*?:\s*([0-9/\s;]+)"
    r".*?Distribui[cç][aã]o.*?:\s*([0-9/;.\s]+)"
    r".*?cycle\s*time.*?:\s*([\w\s;]+)",
    texto, re.S | re.I
)
# Fill the heterogeneous-fleet variables only when the block was found.
if match_heterog:
    # Number of trucks per configuration (group 1).
    variaveis["numero_caminhoes_heterogeneo"] = [
        int(x) for x in re.findall(r"\d+", match_heterog.group(1))
    ]
    # Split group 2 into waste/ore capacity pairs.
    caps = [c.strip() for c in match_heterog.group(2).split(";") if c.strip()]
    esteril, minerio = [], []
    # Split each pair into waste and ore capacity values.
    for c in caps:
        e, m = c.split("/")
        esteril.append(int(e))
        minerio.append(int(m))
    # Store waste and ore capacities of the heterogeneous fleet as lists.
    variaveis["capacity_caminhao_heterogeneo_esteril"] = esteril
    variaveis["capacity_caminhao_heterogeneo_minerio"] = minerio
    # Distribution of the trucks between the two types (group 3), split on ";".
    variaveis["distribuicao_caminhao_heterogeneo"] = [
        x.strip() for x in match_heterog.group(3).split(";") if x.strip()
    ]
    # Cycle-time labels of the heterogeneous fleet (group 4), split on ";".
    variaveis["tempo_ciclo_heterogeneo"] = [
        x.strip() for x in match_heterog.group(4).split(";") if x.strip()
    ]

# Loaders
# Loader block. Groups: (1) loader counts, (2) distribution text, (3) maximum capacity, (4) minimum capacity.
match_carreg = re.search(
    r"Carregadeiras:.*?number[: ]*([0-9; ]+).*?"
    r"distribui[cç][aã]o[: ]*([0-9/; ().]+).*?"
    r"capacidade max[: ]*([0-9; ]+).*?"
    r"capacidade mim[: ]*([0-9; ]+)",
    texto, re.S | re.I
)
# Fill the loader variables only when the block was found.
if match_carreg:
    # Loader counts and capacities: spaces removed, then split on ";" and converted to integers. The distribution is kept as text.
    variaveis["numero_carregadeiras"] = [int(x) for x in match_carreg.group(1).replace(" ", "").split(";") if x]
    variaveis["distribuicao_carregadeiras"] = [x.strip() for x in match_carreg.group(2).split(";") if x.strip()]
    variaveis["capacidade_max_carregadeiras"] = [int(x) for x in match_carreg.group(3).replace(" ", "").split(";") if x]
    variaveis["capacidade_min_carregadeiras"] = [int(x) for x in match_carreg.group(4).replace(" ", "").split(";") if x]

# Fronts and parameters
# Mine fronts: captures the "ore/waste" front-count text after "FRentes:" (e.g. "3/1; 12/5 32/10").
match_frentes = re.search(r"FRentes:.*?Min[ée]rio/Est[ée]ril[: ]*([0-9/; ]+)", texto, re.S | re.I)
if match_frentes:
    # Front-count entries kept as text, split on ";".
    variaveis["frentes"] = [x.strip() for x in match_frentes.group(1).split(";") if x.strip()]

# Control parameters block: captures the numbers after "quantidade" (e.g. "4; 10").
match_param = re.search(r"Par[aâ]metros de controle:.*?quantidade[: ]*([0-9; ]+)", texto, re.S | re.I)
if match_param:
    # Control parameter counts converted to integers.
    variaveis["parametros_controle"] = [int(x) for x in match_param.group(1).replace(" ", "").split(";") if x]

# Discharge points
# Discharge points: captures the "est/min" ratios (e.g. "1/1; 1/3") after "quant (est/min)".
match_descarga = re.search(r"Pontos de descarga:.*?quant.*?\(est/min\)[: ]*([0-9/; ]+)", texto, re.S | re.I)
if match_descarga:
    # Discharge ratios kept as text.
    variaveis["pontos_descarga"] = [x.strip() for x in match_descarga.group(1).split(";") if x.strip()]

# Cycle times
# Cycle-time labels: captures the word characters, commas and spaces after "Tempos de ciclo:" (e.g. "curto, medio, longo").
match_tempo = re.search(r"Tempos de ciclo:.*?([\w, ]+)", texto, re.S | re.I)
if match_tempo:
    # Split the captured text on "," into cycle-time level names.
    variaveis["tempos_ciclo"] = [x.strip() for x in match_tempo.group(1).split(",") if x.strip()]

# === Printing the results ===
print("\n📘 Variáveis extraídas do arquivo instancias.txt:\n")
# Print every parsed parameter as "name = [values]".
for nome, valor in variaveis.items():
    print(f"{nome} = {valor}")


📘 Variáveis extraídas do arquivo instancias.txt:

numero_caminhoes_homogeneo = [30, 40, 50]
capaciti_caminhao_homogeneo_esteril = [61, 120]
capaciti_caminhao_homogeneo_minerio = [64, 130]
tempo_ciclo_homogeneo = ['low', 'avg', 'high\nCam']
numero_caminhoes_heterogeneo = [30, 40, 50]
capacity_caminhao_heterogeneo_esteril = [61, 120]
capacity_caminhao_heterogeneo_minerio = [64, 130]
distribuicao_caminhao_heterogeneo = ['0.5/0.5']
tempo_ciclo_heterogeneo = ['low', 'avg', 'high\nCarregadeiras']
numero_carregadeiras = [3, 6, 9]
distribuicao_carregadeiras = ['0.5/0.5 (6)', '0.34/0.33/0.33 (9)']
capacidade_max_carregadeiras = [800, 900, 1100]
capacidade_min_carregadeiras = [100, 250, 400]
frentes = ['3/1', '12/5 32/10']
parametros_controle = [4, 10]
pontos_descarga = ['1/1', '1/3']
tempos_ciclo = ['curto', 'médio', 'longo']


## Grades - Variables

In [2]:
# Reads the ore-grade tables in teores_csv/ and stores, for each front and each grade row, its list of values in dados_extraidos.
# Keys: "Frente_<min|est>_<k>_<suffix>" for fronts and "teor_<min|rec|max>_<suffix>" for grade rows; the suffix is the front-count text (e.g. 3-1).
import pandas as pd
import numpy as np

# --- 1. CONFIGURATION ---
# Global dictionary to store all the extracted variables
# Global result dictionary, filled by the loop below.
dados_extraidos = {}

# Names of your 3 files
# Grade files to read. The part after " - " in each name is the front suffix (e.g. "3-1" = 3 ore fronts and 1 waste front).
arquivos = [
    'teores_csv/teores.xlsx - 3-1.csv', 
    'teores_csv/teores.xlsx - 12-5.csv', 
    'teores_csv/teores.xlsx - 32-10.csv'
]

# --- 2. FILE PROCESSING ---
# Process each grade file in turn.
for nome_arquivo in arquivos:
    
    # Extraction of the dynamic suffix (e.g. '3-1', '12-5')
    # Default suffix, used when the file name does not follow the "... - <suffix>.csv" pattern.
    sufixo_xy = 'DESCONHECIDO'
    try:
        # Text after the last " - " (e.g. "3-1.csv"), then the part before the first "." (e.g. "3-1").
        sufixo_completo = nome_arquivo.split(' - ')[-1] 
        sufixo_xy = sufixo_completo.split('.')[0] 
    except:
        pass

    print(f"Lendo o arquivo: {nome_arquivo}...")
    df = None
    
    # Robust reading attempt (Excel or CSV)
    
    # Attempt 1: read as Excel (.xlsx)
    # Attempt 1: read as an Excel workbook. The names in this list end in .csv, so this branch is skipped.
    if nome_arquivo.endswith(('.xlsx', '.xls')):
        try:
            df = pd.read_excel(nome_arquivo) 
        except Exception:
            pass # Try the next read
            
    # Attempt 2: read as CSV with COMMA (WITH double quotes, which is the correct one!)
    # Attempt 2: read as CSV with "," as field and decimal separator (values are quoted, e.g. "43,516404").
    if df is None:
        try:
            # FIX: Uses COMMA as the column separator AND as the decimal mark.
            # Pandas will use the double quotes as the field indicator.
            df = pd.read_csv(
                nome_arquivo, 
                sep=',', 
                decimal=',',
                header=0, # Ensures that the first line is the header
                engine='python' # More robust engine
            )
        # If the CSV read also fails, report the error and skip this file.
        except Exception as e:
            # Report the error and skip to the next file
            print(f"❌ ERRO FINAL: Não foi possível ler o arquivo '{nome_arquivo}'. Erro: {e}")
            continue 

    # 3. Cleaning, type conversion and extraction
    if df is not None:
        
        # --- CRITICAL FIX (rename column) ---
        # Ensures that the first column (which holds Frente/teor) is named 'id'
        # The first column holds the row labels (Frente_... and teor_...); it is renamed to "id".
        coluna_id_actual = df.columns[0]
        if coluna_id_actual != 'id':
             df.rename(columns={coluna_id_actual: 'id'}, inplace=True)
             
        # Cleans the remaining column names and converts the values
        # Remove spaces from the column names (e.g. "Massa (t)" becomes "Massa(t)").
        df.columns = df.columns.str.replace(' ', '', regex=False) 
        # Name of the ore-mass column (tonnes); it gets its own conversion below.
        massa_col = 'Massa(t)' if 'Massa(t)' in df.columns else 'Massa(t)'

        # Convert the grade columns to numbers; the id and mass columns are handled separately.
        for col in df.columns:
            if col != 'id' and col != massa_col:
                # The .str.replace('"', '') is not strictly necessary if Pandas works
                # correctly, but we keep it to clean the 'id' column if needed.
                # Remove quotes and spaces from the text, then convert to float; values that cannot be read become NaN.
                df[col] = df[col].astype(str).str.replace('"', '', regex=False).str.replace(' ', '', regex=False)
                df[col] = pd.to_numeric(df[col], errors='coerce')
            # The mass column is only converted to numbers, with the same NaN rule.
            elif col == massa_col:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Extraction
        # Extract the vectors only if the id column was found.
        if 'id' in df.columns:
            df['id'] = df['id'].astype(str)
            
            # Extraction of 'Frente'
            # Distinct ids that start with "Frente" (ore and waste fronts).
            frentes = [col for col in df['id'].unique() if col.startswith('Frente')]
            # For each front: take its row, drop the id and keep the other values as a list (grades in column order, mass last).
            for frente_id in frentes:
                linha_frente = df[df['id'] == frente_id].iloc[0]
                vetor_frente = linha_frente.drop('id').tolist()
                
                # Key of the form Frente_<type>_<k>_<suffix>, e.g. Frente_min_1_3-1.
                nome_variavel = f"{frente_id}_{sufixo_xy}" 
                dados_extraidos[nome_variavel] = vetor_frente

            # Extraction of 'Teor'
            # Same for the grade rows whose id starts with "teor_" (teor_min, teor_rec, teor_max).
            teores = [col for col in df['id'].unique() if col.startswith('teor_')]
            for teor_id in teores:
                linha_teor = df[df['id'] == teor_id].iloc[0]
                vetor_teor = linha_teor.drop('id').tolist()
                
                nome_variavel = f"{teor_id}_{sufixo_xy}"
                dados_extraidos[nome_variavel] = vetor_teor

            # Report how many vectors were stored from this file.
            print(f"✅ SUCESSO! {len(frentes) + len(teores)} vetores extraídos de **{sufixo_xy}**.")
        else:
             print(f"❌ ERRO: A coluna de identificação 'id' não pôde ser resolvida para {sufixo_xy}.")


# --- 4. PRINTING THE RESULTS ---

print("\n" + "#" * 50)
print("# MINAS")
print("#" * 50)

print("="*50)

# Check that at least one vector was extracted and print the outcome.
if not dados_extraidos:
    print("❌ ERRO FINAL: O dicionário 'dados_extraidos' está VAZIO. Algo na leitura/caminho ainda está incorreto.")
    print("="*50)
else:
    print("✅ EXTRAÇÃO CONCLUÍDA. VETORES ARMAZENADOS NO DICIONÁRIO 'dados_extraidos'.")
    print("="*50)

    # Prints the content of the final variables for checking
    # Print every stored vector for checking.
    for nome, vetor in dados_extraidos.items():
        print(f"\n🏷️ **Variável:** {nome}")
        
        vetor_formatado = []
        # Format each value for printing: "NaN" for missing values, floats with up to 6 decimals and no trailing zeros, other values as text.
        for valor in vetor:
            if pd.isna(valor):
                vetor_formatado.append("NaN")
            elif isinstance(valor, float):
                vetor_formatado.append(f"{valor:.6f}".rstrip('0').rstrip('.'))
            else:
                vetor_formatado.append(str(valor))

        # Joins the formatted elements into one clean string
        print(f"[ {', '.join(vetor_formatado)} ]")
        print("-" * 50)

Lendo o arquivo: teores_csv/teores.xlsx - 3-1.csv...
✅ SUCESSO! 7 vetores extraídos de **3-1**.
Lendo o arquivo: teores_csv/teores.xlsx - 12-5.csv...
✅ SUCESSO! 20 vetores extraídos de **12-5**.
Lendo o arquivo: teores_csv/teores.xlsx - 32-10.csv...
✅ SUCESSO! 45 vetores extraídos de **32-10**.

##################################################
# MINAS
##################################################
✅ EXTRAÇÃO CONCLUÍDA. VETORES ARMAZENADOS NO DICIONÁRIO 'dados_extraidos'.

🏷️ **Variável:** Frente_min_1_3-1
[ 43.516404, 35.061517, 1.115758, 0.023837, 0.022763, 1.525551, 6.562331, 48.029205, 26.695373, 1.161455, 8964 ]
--------------------------------------------------

🏷️ **Variável:** Frente_min_2_3-1
[ 57.842206, 14.358051, 0.674921, 0.045281, 0.024178, 1.804572, 17.954423, 61.472359, 7.533439, 0.366785, 57587 ]
--------------------------------------------------

🏷️ **Variável:** Frente_min_3_3-1
[ 65.383658, 1.212513, 0.880919, 0.078981, 0.044143, 3.986686, 79.89528, 65.595105, 

## Cycle time - Variables

In [3]:
# Reads dados/tempo_Ciclo_csv/tempoCiclo_tratado.csv (truck capacities and cycle times per level) and prints its columns.
# The lists built here are the lookup tables used by the .DAT generator in cell 9.
import pandas as pd
import os

# Path of the processed file
# Path of the processed cycle-time file, relative to dados/.
caminho_csv = os.path.join("tempo_Ciclo_csv", "tempoCiclo_tratado.csv")

# === 1. Reads the CSV ===
# Load the CSV into a table with columns Esteril, Minerio, Tempo_ciclo_curto, Tempo_ciclo_medio and Tempo_ciclo_longo.
df = pd.read_csv(caminho_csv)

# === 2. Stores the values in variables ===
# Waste (esteril) truck capacities in tonnes, one value per table row.
capacidade_esteril = df["Esteril"].tolist()
# Ore (minerio) truck capacities in tonnes, same row order as the waste capacities.
capacidade_minerio = df["Minerio"].tolist()
# Cycle times for the three levels (curto = short, medio = medium, longo = long), same row order.
tempo_ciclo_curto = df["Tempo_ciclo_curto"].tolist()
tempo_ciclo_medio = df["Tempo_ciclo_medio"].tolist()
tempo_ciclo_longo = df["Tempo_ciclo_longo"].tolist()

# === 3. Displays all the variables ===
print("📘 Variáveis extraídas de tempoCiclo_tratado.csv:\n")
print(f"capacidade_esteril = {capacidade_esteril}")
print(f"capacidade_minerio = {capacidade_minerio}")
print(f"tempo_ciclo_curto = {tempo_ciclo_curto}")
print(f"tempo_ciclo_medio = {tempo_ciclo_medio}")
print(f"tempo_ciclo_longo = {tempo_ciclo_longo}")

📘 Variáveis extraídas de tempoCiclo_tratado.csv:

capacidade_esteril = [30, 61, 90, 120]
capacidade_minerio = [35, 64, 96, 130]
tempo_ciclo_curto = [5.8, 7.3, 9.8, 14.8]
tempo_ciclo_medio = [7.8, 9.8, 13.13333333, 19.8]
tempo_ciclo_longo = [11.8, 14.8, 19.8, 29.8]


## Production

In [4]:
# Reads producao/teores_producao.csv (production targets and REM) and stores them in dados_mineracao, keyed by discharge ratio ("1/3" or "1/1").
import csv
from pprint import pprint

# --- Configuration ---
# Input path, relative to dados/.
caminho_csv = 'producao/teores_producao.csv' 
# --------------------

# Result dictionary. "1/3": per-destination Minima, Meta and Maxima (BRIT_0..2) plus a total column. "1/1": one destination with Minimo, Meta and Maximo.
# Both entries get a 'rem' list [min, meta, max].
dados_mineracao = {}

try:
    # Read the file as rows of fields and drop the empty rows; the row numbers used below refer to this filtered list.
    with open(caminho_csv, mode='r', encoding='utf-8') as f:
        reader = csv.reader(f)
        rows = [row for row in reader if any(row)]
        
        # Look for the block headers: a row with "Destinos" in column 1 and the ratio ("1/3" or "1/1") in column 2.
        for i, row in enumerate(rows):
            try:
                # --- Block: detection of destination "1/3" ---
                if row[1] == 'Destinos' and row[2] == '1/3':
                    # Destination names (BRIT_0, BRIT_1, BRIT_2) are in columns 4 to 6 of the header row.
                    brit_headers = [h.strip() for h in row[4:7]]
                    # The three rows below the header hold the Minima, Meta and Maxima values.
                    row_minima = rows[i+1]
                    row_meta = rows[i+2]
                    row_maxima = rows[i+3]
                    
                    # Create the entry for the 1/3 ratio.
                    dados_mineracao['1/3'] = {'destinos': {}, 'total': {}}
                    # For each destination, store its Minima, Meta and Maxima (read from the same column as its header).
                    for j, header in enumerate(brit_headers):
                        dados_mineracao['1/3']['destinos'][header] = {
                            'Minima': int(row_minima[j+4]),
                            'Meta': int(row_meta[j+4]),
                            'Maxima': int(row_maxima[j+4])
                        }
                    
                    # Total column (index 7) with the same three values.
                    dados_mineracao['1/3']['total'] = {
                        'Minima': int(row_minima[7]),
                        'Meta': int(row_meta[7]),
                        'Maxima': int(row_maxima[7])
                    }
                    # New REM formatted as [min, meta, max]
                    # REM (waste-to-ore ratio) is hard-coded as [min, meta, max] = [0.25, 0.31, 0.4].
                    dados_mineracao['1/3']['rem'] = [0.25, 0.31, 0.4]

                # --- Block: detection of destination "1/1" ---
                # Same test for the 1/1 ratio; i > 0 makes sure the row above exists.
                if i > 0 and row[1] == 'Destinos' and row[2] == '1/1':
                    # The destination name (BRIT_0) is in column 4 of the row above.
                    brit_header = rows[i-1][4].strip()
                    # This row holds the Minimo value; the next two rows hold Meta and Maximo.
                    row_minimo = row       
                    row_meta = rows[i+1]
                    row_maximo = rows[i+2]
                    
                    # Create the entry for the 1/1 ratio.
                    dados_mineracao['1/1'] = {}
                    # FIX: Changing the target from 2600 to 1600
                    # Store the Minimo, Meta and Maximo of this destination. Meta is set to 1600 here, whatever the file contains.
                    dados_mineracao['1/1'][brit_header] = {
                        'Minimo': int(row_minimo[4]),
                        'Meta': 1600, # Value changed as requested
                        'Maximo': int(row_maximo[4])
                    }
                    # REM hard-coded again, as for the 1/3 ratio.
                    # New REM formatted as [min, meta, max]
                    dados_mineracao['1/1']['rem'] = [0.25, 0.31, 0.4]

            # Rows with a missing column or a non-numeric value are skipped.
            except (IndexError, ValueError) as e:
                continue

    # Print the parsed production data.
    print("--- 📊 Dados de Produção Atualizados ---")
    pprint(dados_mineracao)

# If the file does not exist, print a message and continue.
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado em '{caminho_csv}'")

--- 📊 Dados de Produção Atualizados ---
{'1/1': {'BRIT_0': {'Maximo': 3100, 'Meta': 1600, 'Minimo': 2600},
         'rem': [0.25, 0.31, 0.4]},
 '1/3': {'destinos': {'BRIT_0': {'Maxima': 1300, 'Meta': 1100, 'Minima': 900},
                      'BRIT_1': {'Maxima': 1300, 'Meta': 1100, 'Minima': 900},
                      'BRIT_2': {'Maxima': 1300, 'Meta': 1100, 'Minima': 900}},
         'rem': [0.25, 0.31, 0.4],
         'total': {'Maxima': 3900, 'Meta': 3300, 'Minima': 2700}}}


# Mines

In [5]:
# Generates the Minas/ folder: for each mine line in minas.txt, writes the .DAT input files that the C++ solver reads (src/main.cpp).
# Inputs: grade vectors (cell 3), production targets (cell 7) and cycle-time lookup tables (cell 5).
import os
import re
import math
from collections import defaultdict
import numpy as np 
import unicodedata # Required to remove accents

# --- 1. FUNCTION TO GENERATE THE .DAT FILES (FINAL CORRECTED VERSION) ---

# Writes the .DAT files of every mine in minas.txt. Arguments: dados_extraidos (grades and masses), ARQ_MINAS (path of minas.txt),
# dados_mineracao (production targets) and tc_lookup_tables (cycle-time lookups). Existing files of the same mine are overwritten.
def gerar_arquivos_dat(dados_extraidos, ARQ_MINAS, dados_mineracao, tc_lookup_tables):
    
    # Removes accents and lower-cases a text so it can be compared with plain keywords such as "homogenea".
    def normalizar_frota(texto):
        texto = unicodedata.normalize('NFD', texto).encode('ascii', 'ignore').decode("utf-8")
        return texto.lower().strip()

    # Root output folder, relative to the current directory. The C++ code reads it as ../dados/Minas/ (src/main.cpp).
    PASTA_RAIZ = "Minas"
    os.makedirs(PASTA_RAIZ, exist_ok=True)
    print(f"Criada a pasta raiz: {PASTA_RAIZ}/\n")

    # Cycle-time lookups: for each level (curto, medio, longo), a dictionary from truck capacity to cycle time.
    # time_map_min is keyed by ore capacity (minerio), time_map_est by waste capacity (esteril).
    time_map_min = {}
    time_map_est = {}
    chaves_tempo = ['curto', 'medio', 'longo'] 
    
    # Build both lookups for each level; a missing level is reported and skipped.
    for t_nome in chaves_tempo:
        if t_nome not in tc_lookup_tables:
            print(f"AVISO: Chave de tempo '{t_nome}' não encontrada em tc_lookup_tables.")
            continue
        time_map_min[t_nome] = dict(zip(tc_lookup_tables['cap_min'], tc_lookup_tables[t_nome]))
        time_map_est[t_nome] = dict(zip(tc_lookup_tables['cap_est'], tc_lookup_tables[t_nome]))
    
    # Regex for one line of minas.txt. Groups: (1) mine number, (2) trucks, (3) loaders, (4) fronts as "ore/waste", (5) control parameters,
    # (6) discharge ratio, (7) fleet type (homogenea, homogenea120 or heterogenea), (8) cycle-time level. The rest of the line is ignored.
    padrao = re.compile(
        r"Mina\s*(\d+):\s*(\d+)\s*cam,\s*(\d+)\s*carga,\s*([0-9/]+)\s*frentes,\s*(\d+)\s*par,\s*" +
        r"([0-9/]+)\s*descarga,\s*([\w\d]+(?:120)?),\s*(\w+).*",
        re.IGNORECASE
    )
    
    # Read every line of minas.txt; stop with a message if the file is missing.
    try:
        with open(ARQ_MINAS, "r", encoding="utf-8") as f:
            linhas_minas = f.readlines()
    # Stop with a message if minas.txt does not exist.
    except FileNotFoundError:
        print(f"\n❌ ERRO: Arquivo de minas '{ARQ_MINAS}' não encontrado. Gere o arquivo primeiro.")
        return

    print("="*70)
    print("INICIANDO GERAÇÃO DE PASTAS E ARQUIVOS .DAT PARA CADA MINA")
    print("="*70)
    
    # Keep only the lines that start with "Mina" (the mine data lines), without surrounding whitespace.
    linhas_de_dados_das_minas = [
        linha.strip() for linha in linhas_minas if linha.strip().startswith("Mina")
    ]
    
    # Stop if no mine line was found.
    if not linhas_de_dados_das_minas:
        print(f"❌ ERRO: Nenhuma linha de dados começando com 'Mina' foi encontrada em {ARQ_MINAS}.")
        print("="*70)
        return
        
    print(f"INFO: Encontradas {len(linhas_de_dados_das_minas)} linhas de Mina para processar.\n")

    # Generate the files for each mine line, in the order of minas.txt.
    for linha in linhas_de_dados_das_minas:
        
        # Parse the line with the regex; unrecognised lines are reported and skipped.
        match = padrao.search(linha)
        if not match:
            print(f"AVISO: Linha de Mina não reconhecida (pulando): '{linha}'")
            continue

        # --- LINE PARAMETERS (MINE) ---
        # Mine number and counts: trucks (n_cam), loaders (n_carga) and control parameters (n_par).
        mina_num = int(match.group(1))
        n_cam = int(match.group(2))
        n_carga = int(match.group(3))
        # Fronts as text "ore/waste", e.g. "3/1".
        frentes_par = match.group(4) 
        n_par = int(match.group(5))
        # Discharge ratio as text: "1/1" (one ore destination) or "1/3" (three ore destinations). The waste dump is always added.
        descarga_par = match.group(6).strip()
        # Fleet type and cycle-time level as lower-case text (the level word in minas.txt may carry an accent).
        tipo_frota_original = match.group(7).lower().strip() 
        tempo_ciclo_nome_raw = match.group(8).lower().strip() 
        # Remove the accents of the cycle-time level so it matches the keys curto, medio and longo.
        tempo_ciclo_nome = tempo_ciclo_nome_raw.replace('é', 'e').replace('ê', 'e')
        
        # Fleet type without accents.
        tipo_frota = normalizar_frota(tipo_frota_original) 
        
        # Progress message for this mine.
        print(f"Processando Mina {mina_num} (Desc: {descarga_par}, Frota: {tipo_frota}, Ciclo: {tempo_ciclo_nome_raw})")
        
        # Folder of this mine: Minas/Mina_<N>/, created if it does not exist.
        pasta_mina = os.path.join(PASTA_RAIZ, f"Mina_{mina_num}")
        os.makedirs(pasta_mina, exist_ok=True)
        # Grade suffix of this mine: fronts "3/1" become "3-1", matching the keys built in cell 3.
        teor_sufixo = frentes_par.replace("/", "-")
        
        # === 2. TRUCK CAPACITY ===
        capacidades_min = []
        capacidades_est = []
        
        # Truck capacities in tonnes, hard-coded per fleet type. capacidades_min holds the ore-truck capacities and capacidades_est the waste-truck capacities.
        if 'homogenea120' in tipo_frota:
            # homogenea120: all trucks carry 130 t of ore and 120 t of waste.
            capacidades_min = [130] * n_cam
            capacidades_est = [120] * n_cam
        # homogenea: all trucks carry 64 t of ore and 61 t of waste.
        elif 'homogenea' in tipo_frota:
            capacidades_min = [64] * n_cam
            capacidades_est = [61] * n_cam
        # heterogenea: the first half of the trucks is small (64 t ore, 61 t waste) and the second half is large (130 t ore, 120 t waste).
        elif 'heterogenea' in tipo_frota:
            num_cap_low = n_cam // 2
            num_cap_high = n_cam - num_cap_low
            capacidades_min = [64] * num_cap_low + [130] * num_cap_high
            capacidades_est = [61] * num_cap_low + [120] * num_cap_high
        # Unknown fleet type: no capacity files are written for this mine (its folder was already created).
        else:
             continue
        
        # Write capcam_min_<N>.DAT: one line per truck, "truck_index,ore_capacity".
        with open(os.path.join(pasta_mina, f"capcam_min_{mina_num}.DAT"), "w") as f:
            for c_idx in range(n_cam): f.write(f"{c_idx},{capacidades_min[c_idx]}\n")
        # Write capcam_est_<N>.DAT: one line per truck, "truck_index,waste_capacity".
        with open(os.path.join(pasta_mina, f"capcam_est_{mina_num}.DAT"), "w") as f:
            for c_idx in range(n_cam): f.write(f"{c_idx},{capacidades_est[c_idx]}\n")

        # === 3. LOADER CAPACITY ===
        # Loader productivity bases in t/h, hard-coded with the same values as the loader block of instancias.txt: maximum and minimum per loader class.
        cap_max_base = [800, 900, 1100]
        cap_min_base = [100, 250, 400]
        # The three-value pattern is repeated once per group of three loaders (at least once).
        multiplicador = max(1, n_carga // 3)
        # Maximum and minimum productivity for each loader, cut to n_carga values.
        capacidades_carreg_max = (cap_max_base * multiplicador)[:n_carga]
        capacidades_carreg_min = (cap_min_base * multiplicador)[:n_carga]
            
        # Write cargamax_<N>.DAT: one line per loader, "loader_index,max_productivity".
        with open(os.path.join(pasta_mina, f"cargamax_{mina_num}.DAT"), "w") as f:
            for c_idx in range(n_carga): f.write(f"{c_idx},{capacidades_carreg_max[c_idx]}\n")
        # Write cargamin_<N>.DAT: one line per loader, "loader_index,min_productivity".
        with open(os.path.join(pasta_mina, f"cargamin_{mina_num}.DAT"), "w") as f:
            for c_idx in range(n_carga): f.write(f"{c_idx},{capacidades_carreg_min[c_idx]}\n")

        # === 4. COMPATIBILITY ===
        # Write comp_<N>.DAT: one line per (truck, loader) pair, "truck,loader,compatible", where 1 means compatible and 0 means not compatible.
        with open(os.path.join(pasta_mina, f"comp_{mina_num}.DAT"), "w") as f:
            # Only the heterogeneous fleet has compatibility restrictions; homogeneous fleets are compatible with every loader (comp = 1).
            eh_heterogenea = 'heterogenea' in tipo_frota
            # Split point: the first half of the trucks and the first half of the loaders.
            num_cam_low = n_cam // 2 
            num_ldr_low = n_carga // 2
            for c_idx in range(n_cam):
                for ca_idx in range(n_carga):
                    comp = 1
                    # In the heterogeneous fleet, the first half of the trucks (small ones) is compatible only with the first half of the loaders,
                    # and the second half of the trucks only with the second half of the loaders.
                    if eh_heterogenea:
                        comp = 0
                        if c_idx < num_cam_low: 
                            if ca_idx < num_ldr_low: comp = 1
                        else: 
                            if ca_idx >= num_ldr_low: comp = 1
                    f.write(f"{c_idx},{ca_idx},{comp}\n")

        # === 5. GRADE AND QUANTITY ===
        # Grade data of this mine is looked up with its suffix (e.g. "teor_min_3-1").
        teor_key = f"teor_min_{teor_sufixo}"
        # No grade data for this suffix: skip the rest of this mine (files already written for it are kept).
        if not teor_key in dados_extraidos: continue
             
        # Grade vectors for this suffix: minimum (teor_min), recommended (teor_rec) and maximum (teor_max) grades, one value per element.
        teor_min_vetor = dados_extraidos[f"teor_min_{teor_sufixo}"]
        teor_rec_vetor = dados_extraidos[f"teor_rec_{teor_sufixo}"]
        teor_max_vetor = dados_extraidos[f"teor_max_{teor_sufixo}"]
        
        # Number of ore fronts and waste fronts from the text "ore/waste" (e.g. 3/1 gives 3 and 1); total_frentes is their sum.
        frentes_min_count, frentes_est_count = map(int, frentes_par.split('/'))
        total_frentes = frentes_min_count + frentes_est_count
        
        # Write content_<N>.DAT ("front,element,grade" lines) and quant_<N>.DAT ("front,mass" lines) for every front that has grade data.
        with open(os.path.join(pasta_mina, f"content_{mina_num}.DAT"), "w") as f_content, \
             open(os.path.join(pasta_mina, f"quant_{mina_num}.DAT"), "w") as f_quant:
            # Loop over all fronts of this mine.
            for f_idx in range(total_frentes):
                # The first frentes_min_count fronts are ore fronts (key Frente_min_k); the others are waste fronts (key Frente_est_k). k starts at 1.
                frente_nome_base = f"Frente_min_{f_idx+1}" if f_idx < frentes_min_count else f"Frente_est_{f_idx - frentes_min_count + 1}"
                full_frente_key = f"{frente_nome_base}_{teor_sufixo}"
                # Only fronts that have grade data are written.
                if full_frente_key in dados_extraidos:
                    vetor_frente = dados_extraidos[full_frente_key]
                    # quant file: front index and the mass of the front (the last value of its vector, in tonnes).
                    f_quant.write(f"{f_idx},{vetor_frente[-1]}\n")
                    # content file: one line per control parameter (element) with the grade of this front for that element.
                    for p_idx in range(n_par):
                        f_content.write(f"{f_idx},{p_idx},{vetor_frente[p_idx]}\n")
                        
        # Write cmin_<N>.DAT, crec_<N>.DAT and cmax_<N>.DAT: one line per parameter, "index,value", from the minimum, recommended and maximum grade vectors.
        for nome_arq, vetor in {f"cmin_{mina_num}.DAT": teor_min_vetor, f"crec_{mina_num}.DAT": teor_rec_vetor, f"cmax_{mina_num}.DAT": teor_max_vetor}.items():
             with open(os.path.join(pasta_mina, nome_arq), "w") as f:
                 for p_idx, valor in enumerate(vetor[:n_par]): f.write(f"{p_idx},{valor}\n")

        # === 6. tciclo_# ===
        # Section with cycle times. Number of ore destinations: 1 for "1/1", 3 for "1/3", 0 otherwise. The waste dump is added as the last destination.
        num_destinos_min = 1 if descarga_par == '1/1' else (3 if descarga_par == '1/3' else 0)
        total_destinos = num_destinos_min + 1
        # Index of the waste dump (the last destination).
        idx_destino_esteril = total_destinos - 1 

        # Write tciclo_<N>.DAT: one line per (truck, front, destination), "truck,front,destination,time".
        with open(os.path.join(pasta_mina, f"tciclo_{mina_num}.DAT"), "w") as f_tciclo:
            # Time is the cycle time for matching pairs (ore front with ore destination, waste front with waste dump) and 99999 for the other pairs.
            for c_idx in range(n_cam):
                for f_idx in range(total_frentes):
                    # A front is a waste front when its index is at or after the number of ore fronts.
                    is_f_est = f_idx >= frentes_min_count
                    # Pick the cycle-time lookup for this front type and the capacity of this truck for that type.
                    time_map = time_map_est if is_f_est else time_map_min
                    cap = capacidades_est[c_idx] if is_f_est else capacidades_min[c_idx]
                    # Cycle time for this truck capacity and level; 99999 if the capacity is not in the lookup.
                    base_t = time_map[tempo_ciclo_nome].get(cap, 99999) 
                    for d_idx in range(total_destinos):
                        # The last destination is the waste dump.
                        is_d_est = (d_idx == idx_destino_esteril)
                        # A matching pair (front type equal to destination type) gets the cycle time; a mismatched pair gets 99999.
                        time = base_t if (is_f_est == is_d_est) else 99999
                        f_tciclo.write(f"{c_idx},{f_idx},{d_idx},{time}\n")
        
        # === 7. prod_# (FIX: REM 0.25/0.31/0.4 AND TARGET 1600 FOR 1/1) ===
        # Production file prod_<N>.DAT. Line 1 is the REM as "min, meta, max" = 0.25, 0.31, 0.4.
        with open(os.path.join(pasta_mina, f"prod_{mina_num}.DAT"), "w") as f_prod:
            # First line: minimum, target and maximum REM
            f_prod.write("0.25, 0.31, 0.4\n") 

            # 1/1 mines numbered above 24: writes 2600 as the first value, then the Minimo and Maximo of BRIT_0 from cell 7.
            if descarga_par == '1/1' and mina_num > 24:
                p_data = dados_mineracao['1/1']['BRIT_0']
                # Target changed from 2600 to 1600 as requested
                f_prod.write(f"2600,{p_data['Minimo']},{p_data['Maximo']}\n")
            # 1/1 mines numbered 24 or below: writes 1600 as the first value, then the Minimo and Maximo of BRIT_0 from cell 7.
            elif descarga_par == '1/1' and mina_num <= 24:    
                p_data = dados_mineracao['1/1']['BRIT_0']
                f_prod.write(f"1600,{p_data['Minimo']},{p_data['Maximo']}\n")
            # 1/3 mines: one line per destination (BRIT_0, BRIT_1, BRIT_2) with "Minima,Meta,Maxima" from cell 7.
            elif descarga_par == '1/3':
                d_data = dados_mineracao['1/3']['destinos']
                for brit in ['BRIT_0', 'BRIT_1', 'BRIT_2']:
                    f_prod.write(f"{d_data[brit]['Minima']},{d_data[brit]['Meta']},{d_data[brit]['Maxima']}\n")
        
        # Progress message for this mine's production file.
        print(f"   -> prod_{mina_num}.DAT: REM [0.25, 0.31, 0.4] e Meta {'1600' if descarga_par=='1/1' else 'Padrão'}")
        print("---") 
                     
    print("\n" + "="*70)
    print("✅ GERAÇÃO DE TODOS OS ARQUIVOS .DAT CONCLUÍDA COM SUCESSO.")
    print("="*70)

# --- 2. PREPARATION FOR EXECUTION ---
# Lookup tables for the generator, taken from cell 5: ore capacities (cap_min), waste capacities (cap_est) and the cycle times of the three levels.
tc_lookup_tables = {
    'cap_min': capacidade_minerio,
    'cap_est': capacidade_esteril,
    'curto': tempo_ciclo_curto,
    'medio': tempo_ciclo_medio, 
    'longo': tempo_ciclo_longo
}
# Input file with the mine descriptions, relative to dados/.
ARQ_MINAS = "minas.txt"

# --- 3. RUNNING THE FUNCTION ---
# Run the generator; any exception is printed instead of stopping the notebook.
try:
    gerar_arquivos_dat(dados_extraidos, ARQ_MINAS, dados_mineracao, tc_lookup_tables)
except Exception as e:
    print(f"Erro na execução: {e}")

Criada a pasta raiz: Minas/

INICIANDO GERAÇÃO DE PASTAS E ARQUIVOS .DAT PARA CADA MINA
INFO: Encontradas 290 linhas de Mina para processar.

Processando Mina 1 (Desc: 1/1, Frota: homogenea, Ciclo: curto)
   -> prod_1.DAT: REM [0.25, 0.31, 0.4] e Meta 1600
---
Processando Mina 2 (Desc: 1/1, Frota: heterogenea, Ciclo: curto)
   -> prod_2.DAT: REM [0.25, 0.31, 0.4] e Meta 1600
---
Processando Mina 3 (Desc: 1/1, Frota: homogenea, Ciclo: curto)
   -> prod_3.DAT: REM [0.25, 0.31, 0.4] e Meta 1600
---
Processando Mina 4 (Desc: 1/1, Frota: heterogenea, Ciclo: curto)
   -> prod_4.DAT: REM [0.25, 0.31, 0.4] e Meta 1600
---
Processando Mina 5 (Desc: 1/1, Frota: homogenea120, Ciclo: curto)
   -> prod_5.DAT: REM [0.25, 0.31, 0.4] e Meta 1600
---
Processando Mina 6 (Desc: 1/1, Frota: heterogenea, Ciclo: curto)
   -> prod_6.DAT: REM [0.25, 0.31, 0.4] e Meta 1600
---
Processando Mina 7 (Desc: 1/1, Frota: homogenea120, Ciclo: curto)
   -> prod_7.DAT: REM [0.25, 0.31, 0.4] e Meta 1600
---
Processando M